In [ ]:
import random
import time
from typing import List, Tuple

# =====================================================================
# 1. RANDOMIZED SELECTION (Expected O(n) Time)
# =====================================================================

def partition_3way(arr: List[int], low: int, high: int, pivot: int) -> Tuple[int, int]:
    """
    3-way Dutch National Flag partitioning around a given pivot value.
    Returns indices (lt, gt) such that:
      - arr[low .. lt-1] < pivot
      - arr[lt .. gt] == pivot
      - arr[gt+1 .. high] > pivot
    """
    i = low
    lt = low
    gt = high
    
    while i <= gt:
        if arr[i] < pivot:
            arr[lt], arr[i] = arr[i], arr[lt]
            lt += 1
            i += 1
        elif arr[i] > pivot:
            arr[gt], arr[i] = arr[i], arr[gt]
            gt -= 1
        else:
            i += 1
            
    return lt, gt


def randomized_select(arr: List[int], k: int) -> int:
    """
    Finds the k-th smallest element (0-indexed) in expected O(n) time.
    """
    if not 0 <= k < len(arr):
        raise IndexError("Index k is out of bounds.")
    
    # Work on a copy to prevent mutating the original array
    arr_copy = list(arr)
    return _randomized_select_helper(arr_copy, 0, len(arr_copy) - 1, k)


def _randomized_select_helper(arr: List[int], low: int, high: int, k: int) -> int:
    if low == high:
        return arr[low]
    
    # Select a uniformly random pivot
    pivot_idx = random.randint(low, high)
    pivot = arr[pivot_idx]
    
    lt, gt = partition_3way(arr, low, high, pivot)
    
    if k < lt:
        return _randomized_select_helper(arr, low, lt - 1, k)
    elif k > gt:
        return _randomized_select_helper(arr, gt + 1, high, k)
    else:
        # k falls in the range of elements equal to pivot
        return arr[k]


# =====================================================================
# 2. DETERMINISTIC SELECTION (Worst-Case O(n) Time - Median of Medians)
# =====================================================================

def median_of_medians_select(arr: List[int], k: int) -> int:
    """
    Finds the k-th smallest element (0-indexed) in worst-case O(n) time.
    """
    if not 0 <= k < len(arr):
        raise IndexError("Index k is out of bounds.")
    
    arr_copy = list(arr)
    return _mom_select_helper(arr_copy, 0, len(arr_copy) - 1, k)


def _get_pivot_mom(arr: List[int], low: int, high: int) -> int:
    """
    Recursively computes the Median of Medians for the subarray arr[low..high].
    """
    n = high - low + 1
    if n <= 5:
        # Base case: sort directly and return the median element
        sub = sorted(arr[low:high + 1])
        return sub[n // 2]
    
    # Step 1: Divide array into groups of 5 and find medians
    medians = []
    for i in range(low, high + 1, 5):
        group_high = min(i + 4, high)
        group = sorted(arr[i:group_high + 1])
        medians.append(group[len(group) // 2])
        
    # Step 2: Recursively call _mom_select_helper to find the median of the medians
    return _mom_select_helper(medians, 0, len(medians) - 1, len(medians) // 2)


def _mom_select_helper(arr: List[int], low: int, high: int, k: int) -> int:
    if low == high:
        return arr[low]
    
    # Obtain pivot using Median of Medians algorithm
    pivot = _get_pivot_mom(arr, low, high)
    
    lt, gt = partition_3way(arr, low, high, pivot)
    
    if k < lt:
        return _mom_select_helper(arr, low, lt - 1, k)
    elif k > gt:
        return _mom_select_helper(arr, gt + 1, high, k)
    else:
        return arr[k]


# =====================================================================
# 3. EMPIRICAL BENCHMARKING SCRIPT
# =====================================================================

def run_benchmarks():
    sizes = [1000, 10000, 50000, 100000]
    distributions = ["Random", "Sorted", "Reverse-Sorted", "High Duplicates"]
    
    print(f"{'Size':<10} | {'Distribution':<16} | {'Randomized (s)':<16} | {'Deterministic (s)':<16}")
    print("-" * 68)
    
    for size in sizes:
        for dist in distributions:
            if dist == "Random":
                data = [random.randint(0, 1000000) for _ in range(size)]
            elif dist == "Sorted":
                data = list(range(size))
            elif dist == "Reverse-Sorted":
                data = list(range(size, 0, -1))
            elif dist == "High Duplicates":
                data = [random.randint(0, 10) for _ in range(size)]
                
            k = size // 2  # Find the median element
            
            # Benchmark Randomized
            t0 = time.perf_counter()
            res_rand = randomized_select(data, k)
            t_rand = time.perf_counter() - t0
            
            # Benchmark Median of Medians
            t0 = time.perf_counter()
            res_mom = median_of_medians_select(data, k)
            t_mom = time.perf_counter() - t0
            
            assert res_rand == res_mom, "Mismatch in results!"
            
            print(f"{size:<10} | {dist:<16} | {t_rand:<16.6f} | {t_mom:<16.6f}")

if __name__ == "__main__":
    run_benchmarks()